# LabLens: Custom YOLOv8n Training Pipeline (Colab Ready)
### Tech Elevate 2026 • Team Humpty Dumpty (Mech-ARE)
This notebook fine-tunes an Ultralytics YOLOv8n detector on the 15 core laboratory instruments and exports web-ready TF.js weights for browser execution in LabLens.

In [ ]:
# Step 1: Install Ultralytics and dependencies
!pip install -q ultralytics tensorflow tensorflowjs
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Create dataset.yaml configuration
yaml_content = '''
path: ./dataset
train: images/train
val: images/val
test: images/test

names:
  0: multimeter
  1: caliper
  2: micrometer
  3: oscilloscope
  4: function_generator
  5: power_supply
  6: breadboard
  7: soldering_iron
  8: arduino
  9: raspberry_pi
  10: bunsen_burner
  11: beaker
  12: weighing_scale
  13: tachometer
  14: safety_gear
'''
with open('dataset.yaml', 'w') as f:
    f.write(yaml_content.strip())
print("dataset.yaml written successfully.")

In [ ]:
# Step 3: Train YOLOv8n with photometric augmentations for glare and reflections
from ultralytics import YOLO

# Load lightweight nano model pretrained on COCO
model = YOLO('yolov8n.pt')

# Train on lab instruments dataset
results = model.train(
    data='dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    hsv_h=0.015,     # Color jitter
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,    # Random rotation
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    project='lablens_training',
    name='yolov8n_instruments'
)

In [ ]:
# Step 4: Validate on held-out test split and calculate mAP
metrics = model.val(data='dataset.yaml', split='test')
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

In [ ]:
# Step 5: Export to TF.js format for web browser inference
model.export(format='tfjs', imgsz=640)
print("Model exported to web format. Copy files to public/models/lablens/")